In [0]:
# =============================================================================
# SALLA E-COMMERCE DATA GENERATOR
# =============================================================================
# Generates realistic Saudi e-commerce data for the Salla pipeline demo
# Total: ~16M rows across all tables
# Date range: 2020-01-01 to 2026-04-26
# =============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import shutil
from pyspark.sql.functions import current_timestamp, lit

# Reproducibility
np.random.seed(2026)

# =============================================================================
# PATHS
# =============================================================================
LANDING = "/Volumes/salla_databricks/bronze/landing_data"
BRONZE_CATALOG = "salla_databricks.bronze"

# =============================================================================
# DATE RANGE
# =============================================================================
START_DATE = datetime(2020, 1, 1)
END_DATE = datetime(2026, 4, 26)  # Today

# =============================================================================
# VOLUME TARGETS
# =============================================================================
N_CUSTOMERS = 100_000
N_PRODUCTS = 500
N_STORES = 150
N_ORDERS = 5_000_000
N_REVIEWS = 500_000  # ~10% of orders
N_RETURNS = 250_000  # ~5% of orders

# Orders per year (growth pattern)
ORDERS_PER_YEAR = {
    2020: 300_000,
    2021: 450_000,
    2022: 600_000,
    2023: 800_000,
    2024: 1_000_000,
    2025: 1_100_000,
    2026: 750_000,  # Q1 + Apr only
}

# =============================================================================
# YOY GROWTH MULTIPLIERS (for revenue/ad spend scaling)
# =============================================================================
YOY_MULTIPLIER = {
    2020: 0.40,
    2021: 0.50,
    2022: 0.65,
    2023: 0.80,
    2024: 1.00,
    2025: 1.15,
    2026: 1.25,
}

# =============================================================================
# MONTHLY SEASONALITY
# =============================================================================
MONTH_SEASONALITY = {
    1: 0.85,   # Jan - post holiday slump
    2: 0.80,   # Feb - slow
    3: 1.10,   # Mar - Ramadan shopping
    4: 1.15,   # Apr - Eid Al-Fitr
    5: 0.90,   # May - normal
    6: 0.80,   # Jun - summer start, slow
    7: 0.75,   # Jul - summer, slowest
    8: 0.80,   # Aug - back to school
    9: 0.90,   # Sep - normal
    10: 0.95,  # Oct - pre-white friday
    11: 1.40,  # Nov - WHITE FRIDAY (Saudi Black Friday)
    12: 1.15,  # Dec - year end, holiday shopping
}

# =============================================================================
# SAUDI CITIES & REGIONS
# =============================================================================
SAUDI_CITIES = {
    # City: (Region, Weight)
    "Riyadh": ("Riyadh Region", 0.25),
    "Jeddah": ("Makkah Region", 0.20),
    "Mecca": ("Makkah Region", 0.10),
    "Medina": ("Madinah Region", 0.08),
    "Dammam": ("Eastern Region", 0.10),
    "Khobar": ("Eastern Region", 0.05),
    "Dhahran": ("Eastern Region", 0.03),
    "Taif": ("Makkah Region", 0.04),
    "Tabuk": ("Tabuk Region", 0.03),
    "Buraidah": ("Qassim Region", 0.03),
    "Khamis Mushait": ("Asir Region", 0.03),
    "Abha": ("Asir Region", 0.02),
    "Najran": ("Najran Region", 0.02),
    "Jazan": ("Jazan Region", 0.02),
}
CITY_NAMES = list(SAUDI_CITIES.keys())
CITY_WEIGHTS = [v[1] for v in SAUDI_CITIES.values()]
CITY_TO_REGION = {k: v[0] for k, v in SAUDI_CITIES.items()}

# =============================================================================
# SAUDI NAMES
# =============================================================================
FIRST_NAMES_MALE = ["Mohammed", "Abdullah", "Ahmed", "Khalid", "Fahad", "Saud", "Sultan", 
                   "Faisal", "Turki", "Bandar", "Nawaf", "Majed", "Saad", "Abdulrahman",
                   "Nasser", "Hamad", "Saleh", "Omar", "Ali", "Hassan", "Ibrahim", "Yousef"]
FIRST_NAMES_FEMALE = ["Fatima", "Aisha", "Noura", "Sara", "Maha", "Haya", "Lama", "Dana",
                      "Reem", "Abeer", "Nada", "Dalal", "Haifa", "Mona", "Asma", "Mariam"]
LAST_NAMES = ["Al-Saud", "Al-Rashid", "Al-Dosari", "Al-Ghamdi", "Al-Qahtani", "Al-Harbi",
              "Al-Shehri", "Al-Zahrani", "Al-Otaibi", "Al-Mutairi", "Al-Shamrani", "Al-Maliki",
              "Al-Subaie", "Al-Tamimi", "Al-Anazi", "Al-Juhani", "Al-Bogami", "Al-Yami"]

# =============================================================================
# PRODUCT CATEGORIES WITH REALISTIC SUBCATEGORIES AND BRANDS
# =============================================================================
CATEGORIES = {
    "Fashion": {
        "subcategories": ["Dresses", "Shoes", "Handbags", "Watches", "Abayas", "Thobes", "Scarves", "Jewelry", "Sunglasses"],
        "brands": ["Zara", "H&M", "Mango", "Ounass", "Namshi", "Max Fashion", "Centrepoint", "Splash"],
        "price_range": (50, 2000),
        "price_median": 250,
    },
    "Electronics": {
        "subcategories": ["Smartphones", "Laptops", "Tablets", "Headphones", "Gaming Consoles", "Cameras", "Smart Watches", "TV"],
        "brands": ["Apple", "Samsung", "Sony", "Huawei", "Lenovo", "Dell", "HP", "Xiaomi", "LG"],
        "price_range": (100, 8000),
        "price_median": 1500,
    },
    "Beauty": {
        "subcategories": ["Perfume", "Skincare", "Makeup", "Haircare", "Oud", "Bukhoor", "Body Care", "Nail Care"],
        "brands": ["MAC", "Sephora", "Huda Beauty", "Arabian Oud", "Abdul Samad Al Qurashi", "Bath & Body Works", "Nivea"],
        "price_range": (30, 1500),
        "price_median": 200,
    },
    "Food": {
        "subcategories": ["Arabic Coffee", "Dates", "Honey", "Spices", "Tea", "Snacks", "Chocolates", "Nuts"],
        "brands": ["Al Baik", "Almarai", "Nadec", "Goody", "Al Shifa", "Bateel", "Lipton", "Nestle"],
        "price_range": (15, 300),
        "price_median": 50,
    },
    "Home": {
        "subcategories": ["Furniture", "Lighting", "Kitchenware", "Bedding", "Decor", "Appliances", "Rugs", "Storage"],
        "brands": ["IKEA", "Home Centre", "Pottery Barn", "Crate & Barrel", "Pan Emirates", "Homes R Us"],
        "price_range": (50, 5000),
        "price_median": 400,
    },
    "Sports": {
        "subcategories": ["Running Shoes", "Sportswear", "Fitness Equipment", "Yoga Gear", "Football", "Swimming", "Cycling"],
        "brands": ["Nike", "Adidas", "Puma", "Under Armour", "Reebok", "Decathlon", "Skechers"],
        "price_range": (50, 2000),
        "price_median": 300,
    },
}

VARIANTS = ["Pro", "Elite", "Basic", "Premium", "Plus", "Max", "Lite", "Ultra", "Classic", "Essential"]

# =============================================================================
# ORDER STATUSES & WEIGHTS
# =============================================================================
ORDER_STATUSES = ["delivered", "confirmed", "processing", "shipped", "pending", 
                  "ship_pending", "cancelled", "returned", "refunded"]
ORDER_STATUS_WEIGHTS = [0.45, 0.15, 0.05, 0.05, 0.07, 0.05, 0.08, 0.05, 0.05]

# =============================================================================
# PAYMENT METHODS & GATEWAYS
# =============================================================================
PAYMENT_METHODS = ["visa", "mada", "mastercard", "tamara", "tabby", "stc_pay", "apple_pay", "cash_on_delivery"]
PAYMENT_METHOD_WEIGHTS = [0.22, 0.20, 0.13, 0.10, 0.08, 0.12, 0.05, 0.10]

PAYMENT_GATEWAYS = ["hyperpay", "moyasar", "tap", "payfort", "tamara", "tabby"]
PAYMENT_GATEWAY_WEIGHTS = [0.30, 0.25, 0.20, 0.15, 0.05, 0.05]

PAYMENT_STATUSES = ["paid", "pending", "processing", "failed", "refunded"]
PAYMENT_STATUS_WEIGHTS = [0.70, 0.10, 0.10, 0.05, 0.05]

# =============================================================================
# SHIPPING CARRIERS & STATUSES
# =============================================================================
CARRIERS = ["Aramex", "SMSA", "DHL", "Naqel", "J&T Express", "Fetchr"]
CARRIER_WEIGHTS = [0.30, 0.25, 0.15, 0.15, 0.10, 0.05]

SHIPPING_STATUSES = ["delivered", "in_transit", "pending", "out_for_delivery", 
                    "shipped", "returned", "cancelled"]
SHIPPING_STATUS_WEIGHTS = [0.50, 0.15, 0.10, 0.08, 0.07, 0.05, 0.05]

# =============================================================================
# COMPETITORS (for pricing data)
# =============================================================================
COMPETITORS = [
    {"name": "Amazon.sa", "id": "COMP_AMZ", "country": "SA", "factor": 0.85},  # Cheapest
    {"name": "Noon", "id": "COMP_NON", "country": "SA", "factor": 0.90},
    {"name": "Extra", "id": "COMP_EXT", "country": "SA", "factor": 0.93},
    {"name": "Jarir", "id": "COMP_JAR", "country": "SA", "factor": 1.00},  # Same
    {"name": "Lulu Hypermarket", "id": "COMP_LUL", "country": "SA", "factor": 1.08},  # Most expensive
]

# =============================================================================
# AD PLATFORMS
# =============================================================================
AD_PLATFORMS = {
    "google_ads": {"ctr_base": 0.025, "cpc_base": 1.20, "conv_rate": 0.035},
    "meta": {"ctr_base": 0.018, "cpc_base": 0.80, "conv_rate": 0.028},
    "snapchat": {"ctr_base": 0.015, "cpc_base": 0.50, "conv_rate": 0.020},
    "tiktok": {"ctr_base": 0.035, "cpc_base": 0.60, "conv_rate": 0.025},
}

CAMPAIGN_TYPES = ["awareness", "consideration", "conversion", "retargeting"]
CAMPAIGN_TYPE_WEIGHTS = [0.20, 0.25, 0.35, 0.20]

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def random_date_in_range(start, end, n=1):
    """Generate n random dates between start and end."""
    delta = (end - start).days
    if delta <= 0:
        return [start] * n
    days = np.random.randint(0, delta + 1, n)
    return [start + timedelta(days=int(d)) for d in days]

def inject_nulls(series, pct):
    """Inject NULL values into pct% of a pandas Series."""
    mask = np.random.random(len(series)) < pct
    series = series.copy()
    series[mask] = None
    return series

def inject_bad_values(series, pct, bad_value):
    """Replace pct% of values with bad_value."""
    mask = np.random.random(len(series)) < pct
    series = series.copy()
    series[mask] = bad_value
    return series

def inject_negative(series, pct):
    """Make pct% of numeric values negative."""
    mask = np.random.random(len(series)) < pct
    series = series.copy()
    series[mask] = -abs(series[mask])
    return series

def write_to_landing(df, table_name, partition_cols=None):
    """Write DataFrame to landing zone as CSV."""
    path = f"{LANDING}/{table_name}"
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)
    
    if partition_cols:
        for keys, group in df.groupby(partition_cols):
            if not isinstance(keys, tuple):
                keys = (keys,)
            subpath = "/".join([f"{col}={val}" for col, val in zip(partition_cols, keys)])
            full_path = f"{path}/{subpath}"
            os.makedirs(full_path, exist_ok=True)
            group.drop(columns=partition_cols, errors='ignore').to_csv(f"{full_path}/data.csv", index=False)
    else:
        df.to_csv(f"{path}/data.csv", index=False)
    
    print(f"  ✅ {table_name}: {len(df):,} rows → {path}")

def write_to_bronze(df, table_name):
    """Write DataFrame to Bronze Delta table with _bronze_timestamp."""
    sdf = spark.createDataFrame(df)
    sdf = sdf.withColumn("_bronze_timestamp", current_timestamp())
    full_name = f"{BRONZE_CATALOG}.bronze_{table_name}"
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
    print(f"  ✅ {full_name}: {sdf.count():,} rows")

print("✅ Configuration loaded")
print(f"   Date range: {START_DATE.date()} → {END_DATE.date()}")
print(f"   Target volumes: {N_CUSTOMERS:,} customers, {N_PRODUCTS:,} products, {N_ORDERS:,} orders")

In [0]:
# =============================================================================
# GENERATE 100,000 CUSTOMERS
# =============================================================================
# Noise targets:
#   - 3% NULL email (expect_or_drop) → ~3,000 dropped
#   - 4% bad email format (expect_or_drop) → ~4,000 dropped  
#   - 3% NULL/bad phone (expect) → warning only
#   - 2% NULL name (expect) → warning only
#   - 2% NULL city (expect) → warning only
#   - 1% future created_at (expect) → warning only
# Expected: ~93,000 customers pass to Silver after email validation
# =============================================================================

print("Generating 100,000 customers...")

# Generate clean data first
customer_ids = [f"CUST_{i:06d}" for i in range(1, N_CUSTOMERS + 1)]

# Names - mix of male/female
all_first_names = FIRST_NAMES_MALE + FIRST_NAMES_FEMALE
first_names = np.random.choice(all_first_names, N_CUSTOMERS)
last_names = np.random.choice(LAST_NAMES, N_CUSTOMERS)
customer_names = [f"{fn} {ln}" for fn, ln in zip(first_names, last_names)]

# Emails
email_domains = ["gmail.com", "outlook.sa", "yahoo.com", "hotmail.com", "icloud.com"]
emails = [f"user{i}@{np.random.choice(email_domains)}" for i in range(1, N_CUSTOMERS + 1)]

# Saudi phone numbers (+966 5XXXXXXXX)
phones = [f"+966 5{np.random.randint(10000000, 99999999)}" for _ in range(N_CUSTOMERS)]

# Cities with population weights
cities = np.random.choice(CITY_NAMES, N_CUSTOMERS, p=CITY_WEIGHTS)
regions = [CITY_TO_REGION[c] for c in cities]

# Created dates - exponential distribution (older accounts more common at beginning)
# Range: 2018-01-01 to 2026-04-26
created_days = np.random.exponential(scale=800, size=N_CUSTOMERS).astype(int)
created_days = np.clip(created_days, 0, (END_DATE - datetime(2018, 1, 1)).days)
created_dates = [datetime(2018, 1, 1) + timedelta(days=int(d)) for d in created_days]

# Updated dates - created + random days
updated_dates = [c + timedelta(days=np.random.randint(0, 365)) for c in created_dates]
updated_dates = [min(u, END_DATE) for u in updated_dates]  # Cap at today

# Build DataFrame
customers_df = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_name": customer_names,
    "email": emails,
    "phone": phones,
    "city": cities,
    "region": regions,
    "created_at": [d.strftime("%Y-%m-%d %H:%M:%S") for d in created_dates],
    "updated_at": [d.strftime("%Y-%m-%d %H:%M:%S") for d in updated_dates],
})

# =============================================================================
# INJECT NOISE
# =============================================================================
print("  Injecting noise...")

# 3% NULL email (will be DROPPED by Silver expect_or_drop)
customers_df["email"] = inject_nulls(customers_df["email"], 0.03)

# 4% bad email format (will be DROPPED by Silver expect_or_drop)
bad_email_mask = np.random.random(N_CUSTOMERS) < 0.04
customers_df.loc[bad_email_mask & customers_df["email"].notna(), "email"] = "not_valid_email"

# 3% NULL or short phone (warning only)
phone_noise_mask = np.random.random(N_CUSTOMERS) < 0.03
customers_df.loc[phone_noise_mask, "phone"] = np.where(
    np.random.random(phone_noise_mask.sum()) < 0.5, None, "123"
)

# 2% NULL customer_name (warning only)
customers_df["customer_name"] = inject_nulls(customers_df["customer_name"], 0.02)

# 2% NULL city (warning only)
customers_df["city"] = inject_nulls(customers_df["city"], 0.02)

# 2% NULL region (warning only)
customers_df["region"] = inject_nulls(customers_df["region"], 0.02)

# 1% future created_at (warning only)
future_mask = np.random.random(N_CUSTOMERS) < 0.01
customers_df.loc[future_mask, "created_at"] = "2028-06-15 12:00:00"

# =============================================================================
# WRITE TO LANDING ZONE AND BRONZE
# =============================================================================
print("  Writing to landing zone and Bronze...")
write_to_landing(customers_df, "customers")
write_to_bronze(customers_df, "customers")

# Save for FK references
VALID_CUSTOMER_IDS = customers_df[customers_df["email"].notna() & (customers_df["email"] != "not_valid_email")]["customer_id"].tolist()
ALL_CUSTOMER_IDS = customers_df["customer_id"].tolist()

print(f"\n✅ Customers complete")
print(f"   Total: {len(customers_df):,}")
print(f"   Valid (will pass Silver): ~{len(VALID_CUSTOMER_IDS):,}")
print(f"   Will be dropped (bad email): ~{N_CUSTOMERS - len(VALID_CUSTOMER_IDS):,}")

In [0]:
# =============================================================================
# GENERATE 500 PRODUCTS
# =============================================================================
# Realistic product names with proper brand-category mapping
# Price distribution: lognormal per category
# Noise: 3% negative price, 3% cost>price, 2% NULL name
# =============================================================================

print("Generating 500 products...")

products = []
product_id = 1

# Generate products per category - proportional to market share
category_counts = {
    "Electronics": 100,  # High value, popular
    "Fashion": 120,      # Highest variety
    "Beauty": 80,        # Popular in Saudi
    "Home": 80,          # Growing segment
    "Sports": 60,        # Fitness trend
    "Food": 60,          # Consumables
}

for category, count in category_counts.items():
    cat_info = CATEGORIES[category]
    brands = cat_info["brands"]
    subcats = cat_info["subcategories"]
    price_min, price_max = cat_info["price_range"]
    price_median = cat_info["price_median"]
    
    for _ in range(count):
        brand = np.random.choice(brands)
        subcat = np.random.choice(subcats)
        variant = np.random.choice(VARIANTS)
        
        # Product name: Brand + Subcategory + Variant
        name = f"{brand} {subcat} {variant}"
        
        # Lognormal price distribution centered at category median
        # sigma controls spread
        log_median = np.log(price_median)
        unit_price = np.random.lognormal(mean=log_median, sigma=0.5)
        unit_price = np.clip(unit_price, price_min, price_max)
        unit_price = round(unit_price, 2)
        
        # Cost is 40-70% of price
        cost_ratio = np.random.uniform(0.40, 0.70)
        cost_price = round(unit_price * cost_ratio, 2)
        
        products.append({
            "product_id": f"PROD_{product_id:04d}",
            "product_name": name,
            "category": category,
            "subcategory": subcat,
            "brand": brand,
            "unit_price": unit_price,
            "cost_price": cost_price,
            "current_stock": np.random.randint(0, 1000),
            "reorder_level": np.random.randint(10, 100),
            "product_status": np.random.choice(["active", "inactive", "discontinued"], p=[0.85, 0.10, 0.05]),
        })
        product_id += 1

products_df = pd.DataFrame(products)

# =============================================================================
# INJECT NOISE
# =============================================================================
print("  Injecting noise...")

# 3% negative unit_price
neg_price_mask = np.random.random(len(products_df)) < 0.03
products_df.loc[neg_price_mask, "unit_price"] = -99.99

# 3% cost > unit_price (bad margin)
bad_margin_mask = np.random.random(len(products_df)) < 0.03
products_df.loc[bad_margin_mask, "cost_price"] = products_df.loc[bad_margin_mask, "unit_price"] * 1.5

# 2% NULL product_name
products_df["product_name"] = inject_nulls(products_df["product_name"], 0.02)

# 2% NULL category
products_df["category"] = inject_nulls(products_df["category"], 0.02)

# =============================================================================
# WRITE TO LANDING ZONE AND BRONZE
# =============================================================================
print("  Writing to landing zone and Bronze...")
write_to_landing(products_df, "products")
write_to_bronze(products_df, "products")

# Save for FK references - map product_id to unit_price for order calculation
PRODUCT_PRICES = dict(zip(products_df["product_id"], products_df["unit_price"]))
ALL_PRODUCT_IDS = products_df["product_id"].tolist()
# Weight by popularity - Electronics and Fashion more popular
PRODUCT_WEIGHTS = []
for _, row in products_df.iterrows():
    if row["category"] == "Electronics":
        PRODUCT_WEIGHTS.append(2.0)
    elif row["category"] == "Fashion":
        PRODUCT_WEIGHTS.append(1.5)
    elif row["category"] == "Beauty":
        PRODUCT_WEIGHTS.append(1.3)
    else:
        PRODUCT_WEIGHTS.append(1.0)
PRODUCT_WEIGHTS = np.array(PRODUCT_WEIGHTS) / sum(PRODUCT_WEIGHTS)

print(f"\n✅ Products complete")
print(f"   Total: {len(products_df):,}")
print(f"   Price range: SAR {products_df['unit_price'].min():.2f} - {products_df['unit_price'].max():.2f}")
print(f"   Median price: SAR {products_df['unit_price'].median():.2f}")

In [0]:
# =============================================================================
# GENERATE 150 STORES
# =============================================================================
# Mix of online, physical, and hybrid stores across Saudi cities
# Noise: 3% NULL name, 3% NULL city, 2% NULL store_type
# =============================================================================

print("Generating 150 stores...")

stores = []
for i in range(1, N_STORES + 1):
    city = np.random.choice(CITY_NAMES, p=CITY_WEIGHTS)
    region = CITY_TO_REGION[city]
    store_type = np.random.choice(["online", "physical", "hybrid"], p=[0.40, 0.35, 0.25])
    
    # Store name pattern
    if store_type == "online":
        name = f"Salla Online Store #{i}"
    elif store_type == "physical":
        name = f"Salla {city} Branch #{i}"
    else:
        name = f"Salla {city} Hub #{i}"
    
    stores.append({
        "store_id": f"STORE_{i:04d}",
        "store_name": name,
        "city": city,
        "region": region,
        "store_type": store_type,
        "manager_name": f"{np.random.choice(FIRST_NAMES_MALE)} {np.random.choice(LAST_NAMES)}",
        "opening_date": (datetime(2018, 1, 1) + timedelta(days=np.random.randint(0, 2000))).strftime("%Y-%m-%d"),
        "status": np.random.choice(["active", "inactive"], p=[0.95, 0.05]),
    })

stores_df = pd.DataFrame(stores)

# =============================================================================
# INJECT NOISE
# =============================================================================
print("  Injecting noise...")

# 3% NULL store_name
stores_df["store_name"] = inject_nulls(stores_df["store_name"], 0.03)

# 3% NULL city
stores_df["city"] = inject_nulls(stores_df["city"], 0.03)

# 2% NULL store_type
stores_df["store_type"] = inject_nulls(stores_df["store_type"], 0.02)

# =============================================================================
# WRITE TO LANDING ZONE AND BRONZE
# =============================================================================
print("  Writing to landing zone and Bronze...")
write_to_landing(stores_df, "stores")
write_to_bronze(stores_df, "stores")

# Save for FK references
ALL_STORE_IDS = stores_df["store_id"].tolist()

print(f"\n✅ Stores complete")
print(f"   Total: {len(stores_df):,}")
print(f"   By type: {stores_df['store_type'].value_counts().to_dict()}")

In [0]:
# =============================================================================
# GENERATE 5,000,000 SALES ORDERS
# =============================================================================
# This is the CORE table - everything else links to orders
# Generated in yearly chunks to manage memory
# Uses actual product prices for realistic calculations
# Applies monthly seasonality and YoY growth
#
# Noise targets (~10% across all rules):
#   - 3% NULL customer_id, 2% NULL product_id, 2% NULL store_id (FK violations)
#   - 2% future dates, 2% old dates, 0.5% NULL dates
#   - 2% bad quantity (0 or negative)
#   - 3% invalid status
#   - 2% bad total_amount calculation
#   - 1% negative final_amount
# =============================================================================

print("Generating 5,000,000 sales orders...")
print("  This will take a few minutes...\n")

# We'll store order metadata for payments/shipping/reviews/returns
ORDER_METADATA = []  # List of dicts with order_id, customer_id, product_id, order_date, final_amount

order_counter = 1

# Process year by year and write to Bronze incrementally
first_year = True

for year in range(2020, 2027):
    year_orders = ORDERS_PER_YEAR.get(year, 0)
    if year_orders == 0:
        continue
    
    # Determine max month for this year
    if year == 2026:
        max_month = 4  # Only Jan-Apr 2026
    else:
        max_month = 12
    
    # Distribute orders across months using seasonality
    month_weights = [MONTH_SEASONALITY.get(m, 1.0) for m in range(1, max_month + 1)]
    month_weights = np.array(month_weights) / sum(month_weights)
    orders_per_month = np.random.multinomial(year_orders, month_weights)
    
    print(f"  {year}: {year_orders:,} orders across {max_month} months")
    
    year_orders_list = []
    
    for month_idx, month_order_count in enumerate(orders_per_month):
        month = month_idx + 1
        if month_order_count == 0:
            continue
        
        # Determine days in this month
        if month == 12:
            next_month_start = datetime(year + 1, 1, 1)
        else:
            next_month_start = datetime(year, month + 1, 1)
        month_start = datetime(year, month, 1)
        days_in_month = (next_month_start - month_start).days
        
        # Cap to END_DATE if needed
        if year == 2026 and month == 4:
            days_in_month = 26  # Up to April 26
        
        # Generate order data for this month
        n = month_order_count
        
        # Order IDs
        order_ids = [f"ORD_{order_counter + i:08d}" for i in range(n)]
        order_counter += n
        
        # FKs - from valid lists
        customer_ids = np.random.choice(ALL_CUSTOMER_IDS, n)
        product_ids = np.random.choice(ALL_PRODUCT_IDS, n, p=PRODUCT_WEIGHTS)
        store_ids = np.random.choice(ALL_STORE_IDS, n)
        
        # Order dates - random within month
        days = np.random.randint(1, days_in_month + 1, n)
        order_dates = [datetime(year, month, int(d)) for d in days]
        order_date_strs = [d.strftime("%Y-%m-%d") for d in order_dates]
        
        # Order status
        statuses = np.random.choice(ORDER_STATUSES, n, p=ORDER_STATUS_WEIGHTS)
        
        # Quantity - weighted towards 1-2
        quantities = np.random.choice([1, 2, 3, 4, 5], n, p=[0.50, 0.25, 0.13, 0.07, 0.05])
        
        # Unit prices - look up from product (handle negative prices from noise)
        unit_prices = [max(PRODUCT_PRICES.get(pid, 100), 10) for pid in product_ids]  # Min SAR 10
        unit_prices = np.array(unit_prices)
        
        # Shipping method
        shipping_methods = np.random.choice(
            ["standard", "express", "same_day", "pickup"],
            n, p=[0.50, 0.25, 0.15, 0.10]
        )
        
        # Discount - 0-25%, more in Nov and Ramadan season
        base_discount_pct = np.random.uniform(0, 0.15, n)
        if month == 11:  # White Friday
            base_discount_pct += np.random.uniform(0, 0.10, n)
        elif month in [3, 4]:  # Ramadan/Eid
            base_discount_pct += np.random.uniform(0, 0.05, n)
        base_discount_pct = np.clip(base_discount_pct, 0, 0.25)
        
        # Calculate amounts
        subtotals = quantities * unit_prices
        discount_amounts = np.round(subtotals * base_discount_pct, 2)
        total_amounts = np.round(subtotals - discount_amounts, 2)
        vat_amounts = np.round(total_amounts * 0.15, 2)
        final_amounts = np.round(total_amounts + vat_amounts, 2)
        
        # Build chunk dataframe
        for i in range(n):
            year_orders_list.append({
                "order_id": order_ids[i],
                "customer_id": customer_ids[i],
                "product_id": product_ids[i],
                "store_id": store_ids[i],
                "order_date": order_date_strs[i],
                "order_status": statuses[i],
                "quantity": int(quantities[i]),
                "unit_price": float(unit_prices[i]),
                "discount_amount": float(discount_amounts[i]),
                "total_amount": float(total_amounts[i]),
                "vat_amount": float(vat_amounts[i]),
                "final_amount": float(final_amounts[i]),
                "shipping_method": shipping_methods[i],
                "currency": "SAR",
            })
            
            # Store metadata BEFORE noise injection (for clean FK references)
            ORDER_METADATA.append({
                "order_id": order_ids[i],
                "customer_id": customer_ids[i],
                "product_id": product_ids[i],
                "store_id": store_ids[i],
                "order_date": order_dates[i],
                "final_amount": float(final_amounts[i]),
            })
    
    # Convert year's orders to DataFrame
    year_df = pd.DataFrame(year_orders_list)
    
    # =============================================================================
    # INJECT NOISE (per year chunk)
    # =============================================================================
    n_year = len(year_df)
    
    # 3% NULL customer_id (FK violation)
    year_df["customer_id"] = inject_nulls(year_df["customer_id"], 0.03)
    
    # 2% NULL product_id (FK violation)
    year_df["product_id"] = inject_nulls(year_df["product_id"], 0.02)
    
    # 2% NULL store_id (FK violation)
    year_df["store_id"] = inject_nulls(year_df["store_id"], 0.02)
    
    # 2% future order_date (2028)
    future_mask = np.random.random(n_year) < 0.02
    year_df.loc[future_mask, "order_date"] = "2028-06-15"
    
    # 2% old order_date (2015)
    old_mask = np.random.random(n_year) < 0.02
    year_df.loc[old_mask, "order_date"] = "2015-01-15"
    
    # 0.5% NULL order_date
    year_df["order_date"] = inject_nulls(year_df["order_date"], 0.005)
    
    # 2% bad quantity (0 or -1)
    bad_qty_mask = np.random.random(n_year) < 0.02
    year_df.loc[bad_qty_mask, "quantity"] = np.where(
        np.random.random(bad_qty_mask.sum()) < 0.5, 0, -1
    )
    
    # 2% negative unit_price
    year_df["unit_price"] = inject_negative(year_df["unit_price"].astype(float), 0.02)
    
    # 3% invalid order_status
    invalid_status_mask = np.random.random(n_year) < 0.03
    year_df.loc[invalid_status_mask, "order_status"] = "INVALID_STATUS"
    
    # 2% bad total_amount (set to 999999)
    bad_total_mask = np.random.random(n_year) < 0.02
    year_df.loc[bad_total_mask, "total_amount"] = 999999.0
    
    # 1% negative final_amount
    year_df["final_amount"] = inject_negative(year_df["final_amount"].astype(float), 0.01)
    
    # =============================================================================
    # WRITE YEAR TO LANDING ZONE (partitioned)
    # =============================================================================
    year_df["year"] = year_df["order_date"].str[:4]
    year_df["month"] = year_df["order_date"].str[5:7]
    
    landing_path = f"{LANDING}/sales_orders"
    for (y, m), group in year_df.groupby(["year", "month"]):
        subpath = f"{landing_path}/year={y}/month={m}"
        os.makedirs(subpath, exist_ok=True)
        group.drop(columns=["year", "month"]).to_csv(f"{subpath}/data_{year}.csv", index=False)
    
    # =============================================================================
    # WRITE YEAR TO BRONZE (append or overwrite)
    # =============================================================================
    year_bronze = year_df.drop(columns=["year", "month"])
    sdf = spark.createDataFrame(year_bronze)
    sdf = sdf.withColumn("_bronze_timestamp", current_timestamp())
    
    if first_year:
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BRONZE_CATALOG}.bronze_sales_orders")
        first_year = False
    else:
        sdf.write.mode("append").saveAsTable(f"{BRONZE_CATALOG}.bronze_sales_orders")
    
    print(f"    → Written {n_year:,} orders for {year}")
    
    # Clear memory
    del year_orders_list, year_df, year_bronze, sdf
    import gc
    gc.collect()

print(f"\n✅ Sales orders complete")
print(f"   Total orders: {len(ORDER_METADATA):,}")
print(f"   ORDER_METADATA ready for payments/shipping/reviews/returns")

In [0]:
# =============================================================================
# GENERATE 5,000,000 PAYMENT TRANSACTIONS
# =============================================================================
# 1:1 with orders - each order has exactly one payment
# Amount MUST match order's final_amount
# Includes transaction_id column (required by Silver)
# Processing in chunks to avoid OOM
# =============================================================================

print("Generating 5,000,000 payment transactions...")
print("  Processing in chunks of 500K to avoid OOM...\n")

CHUNK_SIZE = 500_000
total_orders = len(ORDER_METADATA)
first_chunk = True

for chunk_start in range(0, total_orders, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_orders)
    chunk_orders = ORDER_METADATA[chunk_start:chunk_end]
    
    print(f"  Processing chunk {chunk_start:,} - {chunk_end:,}...")
    
    payments = []
    for i, order in enumerate(chunk_orders):
        global_idx = chunk_start + i
        payment_id = f"PAY_{global_idx+1:08d}"
        order_id = order["order_id"]
        order_date = order["order_date"]
        amount = order["final_amount"]
        
        # Payment method and gateway
        payment_method = np.random.choice(PAYMENT_METHODS, p=PAYMENT_METHOD_WEIGHTS)
        payment_gateway = np.random.choice(PAYMENT_GATEWAYS, p=PAYMENT_GATEWAY_WEIGHTS)
        
        # Gateway fee: 1.5-3.5% of amount
        fee_pct = np.random.uniform(0.015, 0.035)
        gateway_fee = round(amount * fee_pct, 2)
        net_amount = round(amount - gateway_fee, 2)
        
        # Transaction date: same day or +1 day
        txn_date = order_date + timedelta(days=int(np.random.choice([0, 0, 0, 1])))
        
        # Settlement date: +1 to +7 days after transaction
        settlement_date = txn_date + timedelta(days=int(np.random.randint(1, 8)))
        
        # Status
        payment_status = np.random.choice(PAYMENT_STATUSES, p=PAYMENT_STATUS_WEIGHTS)
        settlement_status = np.random.choice(["settled", "pending", "failed"], p=[0.70, 0.20, 0.10])
        
        payments.append({
            "payment_id": payment_id,
            "transaction_id": payment_id,  # Silver references this column
            "order_id": order_id,
            "payment_method": payment_method,
            "payment_gateway": payment_gateway,
            "amount": amount,
            "gateway_fee": gateway_fee,
            "net_amount": net_amount,
            "currency": "SAR",
            "payment_status": payment_status,
            "settlement_status": settlement_status,
            "transaction_date": txn_date.strftime("%Y-%m-%d"),
            "settlement_date": settlement_date.strftime("%Y-%m-%d"),
        })
    
    payments_df = pd.DataFrame(payments)
    n_chunk = len(payments_df)
    
    # =============================================================================
    # INJECT NOISE
    # =============================================================================
    # 3% NULL payment_id (will be DROPPED by expect_or_drop)
    payments_df["payment_id"] = inject_nulls(payments_df["payment_id"], 0.03)
    
    # 2% NULL order_id
    payments_df["order_id"] = inject_nulls(payments_df["order_id"], 0.02)
    
    # 3% negative amount
    payments_df["amount"] = inject_negative(payments_df["amount"].astype(float), 0.03)
    
    # 3% negative gateway_fee
    payments_df["gateway_fee"] = inject_negative(payments_df["gateway_fee"].astype(float), 0.03)
    
    # 3% bad net_amount (set to 999999)
    bad_net_mask = np.random.random(n_chunk) < 0.03
    payments_df.loc[bad_net_mask, "net_amount"] = 999999.0
    
    # 3% invalid payment_status
    invalid_mask = np.random.random(n_chunk) < 0.03
    payments_df.loc[invalid_mask, "payment_status"] = "INVALID_STATUS"
    
    # 2% NULL payment_method
    payments_df["payment_method"] = inject_nulls(payments_df["payment_method"], 0.02)
    
    # 1% future transaction_date
    future_mask = np.random.random(n_chunk) < 0.01
    payments_df.loc[future_mask, "transaction_date"] = "2030-01-01"
    
    # =============================================================================
    # WRITE CHUNK TO LANDING ZONE
    # =============================================================================
    landing_path = f"{LANDING}/payment_transactions"
    if chunk_start == 0:
        shutil.rmtree(landing_path, ignore_errors=True)
        os.makedirs(landing_path, exist_ok=True)
    payments_df.to_csv(f"{landing_path}/data_{chunk_start}.csv", index=False)
    
    # =============================================================================
    # WRITE CHUNK TO BRONZE
    # =============================================================================
    sdf = spark.createDataFrame(payments_df)
    sdf = sdf.withColumn("_bronze_timestamp", current_timestamp())
    
    if first_chunk:
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BRONZE_CATALOG}.bronze_payment_transactions")
        first_chunk = False
    else:
        sdf.write.mode("append").saveAsTable(f"{BRONZE_CATALOG}.bronze_payment_transactions")
    
    print(f"    → Written {n_chunk:,} payments")
    
    # Clear memory
    del payments, payments_df, sdf
    import gc
    gc.collect()

print(f"\n✅ Payment transactions complete")
print(f"   Total: {total_orders:,}")

In [0]:
# =============================================================================
# GENERATE 5,000,000 SHIPPING DETAILS
# =============================================================================
# 1:1 with orders - each order has exactly one shipping record
# Includes all columns required by Silver
# Processing in chunks to avoid OOM
# =============================================================================

print("Generating 5,000,000 shipping details...")
print("  Processing in chunks of 500K to avoid OOM...\n")

# Build customer lookup for delivery addresses
customer_cities = dict(zip(customers_df["customer_id"], customers_df["city"]))
customer_regions = dict(zip(customers_df["customer_id"], customers_df["region"]))

CHUNK_SIZE = 500_000
total_orders = len(ORDER_METADATA)
first_chunk = True

for chunk_start in range(0, total_orders, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_orders)
    chunk_orders = ORDER_METADATA[chunk_start:chunk_end]
    
    print(f"  Processing chunk {chunk_start:,} - {chunk_end:,}...")
    
    shipping = []
    for i, order in enumerate(chunk_orders):
        global_idx = chunk_start + i
        shipping_id = f"SHIP_{global_idx+1:08d}"
        order_id = order["order_id"]
        customer_id = order["customer_id"]
        order_date = order["order_date"]
        
        # Carrier
        carrier = np.random.choice(CARRIERS, p=CARRIER_WEIGHTS)
        
        # Shipping cost based on carrier and random
        base_cost = np.random.uniform(15, 50)
        if carrier in ["DHL", "Aramex"]:
            base_cost *= 1.3  # Premium carriers
        shipping_cost = round(base_cost, 2)
        
        # Shipped date: order_date + 0-2 days
        shipped_date = order_date + timedelta(days=int(np.random.choice([0, 1, 1, 2])))
        
        # Estimated delivery: shipped + 3-7 days
        est_days = int(np.random.randint(3, 8))
        estimated_delivery = shipped_date + timedelta(days=est_days)
        
        # Actual delivery: shipped + 1-14 days
        actual_days = int(np.random.choice(
            [1, 2, 3, 4, 5, 6, 7, 8, 10, 14],
            p=[0.05, 0.10, 0.20, 0.25, 0.20, 0.10, 0.05, 0.03, 0.01, 0.01]
        ))
        actual_delivery = shipped_date + timedelta(days=actual_days)
        
        # Delivery days
        delivery_days = (actual_delivery - shipped_date).days
        
        # Tracking number
        tracking_number = f"TRK{np.random.randint(100000000, 999999999)}"
        
        # Delivery address from customer
        delivery_city = customer_cities.get(customer_id, "Riyadh")
        delivery_region = customer_regions.get(customer_id, "Riyadh Region")
        delivery_address = f"Street {np.random.randint(1, 999)}, Building {np.random.randint(1, 100)}"
        
        # Status
        shipping_status = np.random.choice(SHIPPING_STATUSES, p=SHIPPING_STATUS_WEIGHTS)
        
        shipping.append({
            "shipping_id": shipping_id,
            "order_id": order_id,
            "carrier": carrier,
            "tracking_number": tracking_number,
            "shipping_status": shipping_status,
            "shipping_cost": shipping_cost,
            "shipped_date": shipped_date.strftime("%Y-%m-%d"),
            "estimated_delivery": estimated_delivery.strftime("%Y-%m-%d"),
            "actual_delivery": actual_delivery.strftime("%Y-%m-%d"),
            "delivery_days": delivery_days,
            "delivery_address": delivery_address,
            "delivery_city": delivery_city if delivery_city else "Riyadh",
            "delivery_region": delivery_region if delivery_region else "Riyadh Region",
        })
    
    shipping_df = pd.DataFrame(shipping)
    n_chunk = len(shipping_df)
    
    # =============================================================================
    # INJECT NOISE
    # =============================================================================
    # 3% NULL shipping_id (will be DROPPED by expect_or_drop)
    shipping_df["shipping_id"] = inject_nulls(shipping_df["shipping_id"], 0.03)
    
    # 2% NULL order_id
    shipping_df["order_id"] = inject_nulls(shipping_df["order_id"], 0.02)
    
    # 3% negative shipping_cost
    shipping_df["shipping_cost"] = inject_negative(shipping_df["shipping_cost"].astype(float), 0.03)
    
    # 4% invalid shipping_status
    invalid_mask = np.random.random(n_chunk) < 0.04
    shipping_df.loc[invalid_mask, "shipping_status"] = "GARBAGE_STATUS"
    
    # 2% future shipped_date
    future_mask = np.random.random(n_chunk) < 0.02
    shipping_df.loc[future_mask, "shipped_date"] = "2030-01-01"
    
    # 2% actual_delivery before shipped_date (illogical)
    bad_date_mask = np.random.random(n_chunk) < 0.02
    shipping_df.loc[bad_date_mask, "delivery_days"] = -5
    
    # 1% NULL carrier
    shipping_df["carrier"] = inject_nulls(shipping_df["carrier"], 0.01)
    
    # =============================================================================
    # WRITE CHUNK TO LANDING ZONE
    # =============================================================================
    landing_path = f"{LANDING}/shipping_details"
    if chunk_start == 0:
        shutil.rmtree(landing_path, ignore_errors=True)
        os.makedirs(landing_path, exist_ok=True)
    shipping_df.to_csv(f"{landing_path}/data_{chunk_start}.csv", index=False)
    
    # =============================================================================
    # WRITE CHUNK TO BRONZE
    # =============================================================================
    sdf = spark.createDataFrame(shipping_df)
    sdf = sdf.withColumn("_bronze_timestamp", current_timestamp())
    
    if first_chunk:
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BRONZE_CATALOG}.bronze_shipping_details")
        first_chunk = False
    else:
        sdf.write.mode("append").saveAsTable(f"{BRONZE_CATALOG}.bronze_shipping_details")
    
    print(f"    → Written {n_chunk:,} shipping records")
    
    # Clear memory
    del shipping, shipping_df, sdf
    import gc
    gc.collect()

print(f"\n✅ Shipping details complete")
print(f"   Total: {total_orders:,}")

In [0]:
# =============================================================================
# GENERATE 500,000 PRODUCT REVIEWS
# =============================================================================
# ~10% of orders get reviewed
# Reviews link to ACTUAL order's customer_id and product_id
# Review date is AFTER order date
# Includes order_id column (required by Silver)
#
# Noise targets:
#   - 2% NULL review_id
#   - 2% invalid customer_id
#   - 1% rating = 99 (out of range)
#   - 1% future review_date
# =============================================================================

print("Generating 500,000 product reviews...")

# Sample 500K random orders for reviews
review_orders = np.random.choice(len(ORDER_METADATA), size=N_REVIEWS, replace=False)

REVIEW_TEXTS = [
    "Excellent product, highly recommended!",
    "Good quality for the price.",
    "Average product, nothing special.",
    "Not what I expected, disappointed.",
    "Amazing quality, will buy again!",
    "Fast delivery, great packaging.",
    "Product matches the description perfectly.",
    "Could be better for this price.",
    "Love it! My family is happy.",
    "Shipping was slow but product is good.",
    "ممتاز جداً، أنصح به",  # Arabic: Excellent, I recommend it
    "جودة عالية وسعر مناسب",  # Arabic: High quality and good price
    "التوصيل كان سريع",  # Arabic: Delivery was fast
]

reviews = []
for i, order_idx in enumerate(review_orders):
    if i % 100_000 == 0:
        print(f"  Processing {i:,} / {N_REVIEWS:,}...")
    
    order = ORDER_METADATA[order_idx]
    
    review_id = f"REV_{i+1:07d}"
    order_id = order["order_id"]
    product_id = order["product_id"]
    customer_id = order["customer_id"]
    order_date = order["order_date"]
    
    # Review date: 3-30 days after order
    review_date = order_date + timedelta(days=np.random.randint(3, 31))
    
    # Rating: weighted towards positive
    rating = np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.30, 0.35])
    
    # Review text based on rating
    if rating >= 4:
        text = np.random.choice(REVIEW_TEXTS[:7])
    elif rating == 3:
        text = np.random.choice(REVIEW_TEXTS[2:5])
    else:
        text = np.random.choice(REVIEW_TEXTS[3:5])
    
    reviews.append({
        "review_id": review_id,
        "product_id": product_id,
        "customer_id": customer_id,
        "order_id": order_id,  # Required by Silver
        "review_date": review_date.strftime("%Y-%m-%d"),
        "rating": rating,
        "review_text": text,
        "verified_purchase": np.random.choice([True, False], p=[0.85, 0.15]),
        "helpful_votes": np.random.randint(0, 100),
        "review_source": np.random.choice(["app", "web", "email"], p=[0.50, 0.40, 0.10]),
    })

reviews_df = pd.DataFrame(reviews)
print(f"  Total reviews before noise: {len(reviews_df):,}")

# =============================================================================
# INJECT NOISE
# =============================================================================
print("  Injecting noise...")

n_reviews = len(reviews_df)

# 2% NULL review_id
reviews_df["review_id"] = inject_nulls(reviews_df["review_id"], 0.02)

# 2% invalid customer_id
invalid_cust_mask = np.random.random(n_reviews) < 0.02
reviews_df.loc[invalid_cust_mask, "customer_id"] = "CUST_INVALID"

# 1% rating = 99 (out of range)
bad_rating_mask = np.random.random(n_reviews) < 0.01
reviews_df.loc[bad_rating_mask, "rating"] = 99

# 1% future review_date
future_mask = np.random.random(n_reviews) < 0.01
reviews_df.loc[future_mask, "review_date"] = "2030-01-01"

# =============================================================================
# WRITE TO LANDING ZONE AND BRONZE
# =============================================================================
print("  Writing to landing zone and Bronze...")
write_to_landing(reviews_df, "product_reviews")
write_to_bronze(reviews_df, "product_reviews")

print(f"\n✅ Product reviews complete")
print(f"   Total: {len(reviews_df):,}")
print(f"   Average rating: {reviews_df['rating'].mean():.2f}")

In [0]:
# =============================================================================
# GENERATE 250,000 CUSTOMER RETURNS
# =============================================================================
# ~5% of orders get returned
# Returns link to ACTUAL order's customer_id, product_id, order_id
# Return date is AFTER order date
# Refund amount is portion of order's final_amount
#
# Noise targets:
#   - 2% NULL return_id
#   - 1% negative refund_amount
#   - 1% invalid return_status
# =============================================================================

print("Generating 250,000 customer returns...")

# Prefer orders with cancelled/returned/refunded status
return_candidates = []
for i, order in enumerate(ORDER_METADATA):
    # Use orders_df to check status (it has noise injected, but that's fine)
    return_candidates.append(i)

# Sample 250K orders for returns
return_orders = np.random.choice(return_candidates, size=N_RETURNS, replace=False)

RETURN_REASONS = ["defective", "wrong_item", "changed_mind", "damaged", "late_delivery", "not_as_described"]
RETURN_REASON_WEIGHTS = [0.20, 0.15, 0.30, 0.15, 0.10, 0.10]

RETURN_STATUSES = ["approved", "completed", "pending", "rejected"]
RETURN_STATUS_WEIGHTS = [0.35, 0.35, 0.20, 0.10]

REFUND_METHODS = ["original_payment", "store_credit", "bank_transfer"]
REFUND_METHOD_WEIGHTS = [0.60, 0.25, 0.15]

returns = []
for i, order_idx in enumerate(return_orders):
    if i % 50_000 == 0:
        print(f"  Processing {i:,} / {N_RETURNS:,}...")
    
    order = ORDER_METADATA[order_idx]
    
    return_id = f"RET_{i+1:07d}"
    order_id = order["order_id"]
    product_id = order["product_id"]
    customer_id = order["customer_id"]
    order_date = order["order_date"]
    final_amount = order["final_amount"]
    
    # Return date: 3-30 days after order
    return_date = order_date + timedelta(days=np.random.randint(3, 31))
    
    # Refund amount: 50-100% of order amount
    refund_pct = np.random.uniform(0.50, 1.00)
    refund_amount = round(final_amount * refund_pct, 2)
    
    returns.append({
        "return_id": return_id,
        "order_id": order_id,
        "customer_id": customer_id,
        "product_id": product_id,
        "return_date": return_date.strftime("%Y-%m-%d"),
        "return_reason": np.random.choice(RETURN_REASONS, p=RETURN_REASON_WEIGHTS),
        "return_status": np.random.choice(RETURN_STATUSES, p=RETURN_STATUS_WEIGHTS),
        "refund_amount": refund_amount,
        "refund_method": np.random.choice(REFUND_METHODS, p=REFUND_METHOD_WEIGHTS),
        "processed_date": (return_date + timedelta(days=np.random.randint(1, 10))).strftime("%Y-%m-%d"),
    })

returns_df = pd.DataFrame(returns)
print(f"  Total returns before noise: {len(returns_df):,}")

# =============================================================================
# INJECT NOISE
# =============================================================================
print("  Injecting noise...")

n_returns = len(returns_df)

# 2% NULL return_id
returns_df["return_id"] = inject_nulls(returns_df["return_id"], 0.02)

# 1% negative refund_amount
returns_df["refund_amount"] = inject_negative(returns_df["refund_amount"].astype(float), 0.01)

# 1% invalid return_status
invalid_mask = np.random.random(n_returns) < 0.01
returns_df.loc[invalid_mask, "return_status"] = "INVALID"

# =============================================================================
# WRITE TO LANDING ZONE AND BRONZE
# =============================================================================
print("  Writing to landing zone and Bronze...")
write_to_landing(returns_df, "customer_returns")
write_to_bronze(returns_df, "customer_returns")

print(f"\n✅ Customer returns complete")
print(f"   Total: {len(returns_df):,}")
print(f"   Total refund value: SAR {returns_df['refund_amount'].sum():,.0f}")

In [0]:
# =============================================================================
# GENERATE 50,000 AD SPEND RECORDS
# =============================================================================
# Daily records per platform × category, 2020-2026
# Platforms: Google Ads, Meta, Snapchat, TikTok
# Categories: All 6 product categories
# Every 3rd day to manage volume
#
# Metrics scale with YoY growth and monthly seasonality
# ROAS calculated realistically (revenue / spend)
# =============================================================================

print("Generating ad spend records (2020-2026)...")

platforms = list(AD_PLATFORMS.keys())
categories = list(CATEGORIES.keys())

ad_spend_records = []
ad_id = 1

current_date = datetime(2020, 1, 1)
while current_date <= END_DATE:
    year = current_date.year
    month = current_date.month
    
    # Year and month multipliers
    year_mult = YOY_MULTIPLIER.get(year, 1.0)
    month_mult = MONTH_SEASONALITY.get(month, 1.0)
    
    for platform in platforms:
        platform_config = AD_PLATFORMS[platform]
        
        for category in categories:
            # Base impressions scale with year and month
            base_impressions = np.random.randint(50000, 200000)
            impressions = int(base_impressions * year_mult * month_mult * np.random.uniform(0.8, 1.2))
            
            # CTR with some randomness
            ctr = platform_config["ctr_base"] * np.random.uniform(0.7, 1.3)
            clicks = int(impressions * ctr)
            
            # CPC varies by category (Electronics higher, Food lower)
            category_cpc_mult = {"Electronics": 1.5, "Beauty": 1.2, "Fashion": 1.1, "Home": 1.0, "Sports": 0.9, "Food": 0.7}
            cpc = platform_config["cpc_base"] * category_cpc_mult.get(category, 1.0) * np.random.uniform(0.8, 1.2)
            
            # Spend
            spend = round(clicks * cpc, 2)
            
            # Conversions
            conv_rate = platform_config["conv_rate"] * np.random.uniform(0.6, 1.4)
            conversions = int(clicks * conv_rate)
            
            # Revenue from conversions (use category median as AOV)
            aov = CATEGORIES[category]["price_median"] * 1.15  # Including VAT
            revenue = round(conversions * aov * year_mult * np.random.uniform(0.8, 1.2), 2)
            
            # ROAS
            roas = round(revenue / spend, 2) if spend > 0 else 0
            
            # CPM
            cpm = round((spend / impressions) * 1000, 2) if impressions > 0 else 0
            
            # Campaign type
            campaign_type = np.random.choice(CAMPAIGN_TYPES, p=CAMPAIGN_TYPE_WEIGHTS)
            
            ad_spend_records.append({
                "ad_spend_id": f"AD_{ad_id:06d}",
                "date": current_date.strftime("%Y-%m-%d"),
                "platform": platform,
                "category": category,
                "campaign_type": campaign_type,
                "campaign_name": f"{platform.title()}_{category}_{campaign_type}_{current_date.strftime('%Y%m')}",
                "impressions": impressions,
                "clicks": clicks,
                "spend_sar": spend,
                "conversions": conversions,
                "revenue_sar": revenue,
                "roas": roas,
                "ctr": round(ctr * 100, 2),  # As percentage
                "cpc": round(cpc, 2),
                "cpm": cpm,
                "currency": "SAR",
                "year": year,
                "month": month,
                "day": current_date.day,
            })
            ad_id += 1
    
    # Every 3rd day
    current_date += timedelta(days=3)

ad_spend_df = pd.DataFrame(ad_spend_records)
print(f"  Total ad spend records: {len(ad_spend_df):,}")

# =============================================================================
# WRITE TO LANDING ZONE ONLY (Auto Loader will pick these up)
# =============================================================================
print("  Writing to landing zone (partitioned by year/month/day)...")

# Clear existing ad_spend folder
ad_spend_path = f"{LANDING}/ad_spend"
shutil.rmtree(ad_spend_path, ignore_errors=True)

write_to_landing(ad_spend_df, "ad_spend", partition_cols=["year", "month", "day"])

print(f"\n✅ Ad spend complete")
print(f"   Total: {len(ad_spend_df):,}")
print(f"   Total spend: SAR {ad_spend_df['spend_sar'].sum():,.0f}")
print(f"   Avg ROAS: {ad_spend_df['roas'].mean():.2f}")

In [0]:
# =============================================================================
# GENERATE 30,000 COMPETITOR PRICING RECORDS
# =============================================================================
# Weekly scrapes, 2020-2026
# 5 competitors with different pricing strategies
# CRITICAL: product_name MUST MATCH exactly with dim_products for Gold joins
#
# Price factors:
#   Amazon.sa: 0.85x (cheapest)
#   Noon: 0.90x
#   Extra: 0.93x
#   Jarir: 1.00x (same price)
#   Lulu: 1.08x (most expensive)
# =============================================================================

print("Generating competitor pricing records (2020-2026)...")

# Use ACTUAL product names from products_df (excluding NULL names)
valid_products = products_df[products_df["product_name"].notna()].copy()
product_price_lookup = dict(zip(valid_products["product_name"], valid_products["unit_price"]))
product_category_lookup = dict(zip(valid_products["product_name"], valid_products["category"]))
product_subcat_lookup = dict(zip(valid_products["product_name"], valid_products["subcategory"]))

# Sample 100 products for competitor tracking
tracked_products = valid_products["product_name"].sample(min(100, len(valid_products))).tolist()
print(f"  Tracking {len(tracked_products)} products across {len(COMPETITORS)} competitors")

competitor_records = []
scrape_id = 1

current_date = datetime(2020, 1, 1)
while current_date <= END_DATE:
    year = current_date.year
    month = current_date.month
    
    for comp in COMPETITORS:
        # Random subset of tracked products for this scrape (70-100%)
        n_products = int(len(tracked_products) * np.random.uniform(0.7, 1.0))
        scrape_products = np.random.choice(tracked_products, size=n_products, replace=False)
        
        for product_name in scrape_products:
            our_price = product_price_lookup.get(product_name, 100)
            if our_price < 0:  # Skip negative prices from noise
                our_price = 100
            
            # Competitor's current price
            price_factor = comp["factor"] * np.random.uniform(0.85, 1.15)
            current_price = round(our_price * price_factor, 2)
            
            # Original price (before any discount)
            original_price = round(current_price * np.random.uniform(1.0, 1.35), 2)
            
            # Discount percentage
            discount_pct = round((1 - current_price / original_price) * 100, 1) if original_price > current_price else 0
            
            competitor_records.append({
                "scrape_id": scrape_id,
                "scrape_timestamp": f"{current_date.strftime('%Y-%m-%d')}T{np.random.randint(6, 22):02d}:{np.random.randint(0, 60):02d}:00",
                "scrape_date": current_date.strftime("%Y-%m-%d"),
                "competitor_id": comp["id"],
                "competitor_name": comp["name"],
                "competitor_country": comp["country"],
                "product_name": product_name,  # EXACT MATCH with dim_products
                "product_category": product_category_lookup.get(product_name, "Unknown"),
                "product_subcategory": product_subcat_lookup.get(product_name, "Unknown"),
                "current_price": current_price,
                "original_price": original_price,
                "discount_percentage": discount_pct,
                "currency": "SAR",
                "in_stock": np.random.choice([True, False], p=[0.85, 0.15]),
                "rating": round(np.random.uniform(3.0, 5.0), 1),
                "review_count": np.random.randint(10, 5000),
                "year": year,
                "month": month,
                "day": current_date.day,
            })
            scrape_id += 1
    
    # Weekly scrapes
    current_date += timedelta(days=7)

competitor_df = pd.DataFrame(competitor_records)
print(f"  Total competitor records: {len(competitor_df):,}")

# =============================================================================
# WRITE TO LANDING ZONE ONLY (Auto Loader will pick these up)
# =============================================================================
print("  Writing to landing zone (partitioned by year/month/day)...")

# Clear existing competitor_pricing folder
comp_path = f"{LANDING}/competitor_pricing"
shutil.rmtree(comp_path, ignore_errors=True)

write_to_landing(competitor_df, "competitor_pricing", partition_cols=["year", "month", "day"])

print(f"\n✅ Competitor pricing complete")
print(f"   Total: {len(competitor_df):,}")
print(f"   Competitors: {competitor_df['competitor_name'].nunique()}")
print(f"   Products tracked: {competitor_df['product_name'].nunique()}")

In [0]:
# =============================================================================
# GENERATE SUPPORTING LANDING ZONE TABLES (VECTORIZED)
# =============================================================================
print("Generating supporting landing zone tables...")

# =============================================================================
# 1. EXCHANGE RATES
# =============================================================================
print("\n1. Exchange rates...")
date_range_pd = pd.date_range(START_DATE, END_DATE, freq='D')
currencies = {"USD": 3.75, "EUR": 4.10, "GBP": 4.75, "AED": 1.02}

exchange_rows = []
for cur, base in currencies.items():
    for d in date_range_pd:
        exchange_rows.append({"from_currency": cur, "to_currency": "SAR",
            "exchange_rate": round(base * np.random.uniform(0.98, 1.02), 4),
            "rate_date": d.strftime("%Y-%m-%d"),
            "source": np.random.choice(["saudi_central_bank", "xe.com", "bloomberg"]),
            "year": d.year, "month": d.month, "day": d.day})
exchange_df = pd.DataFrame(exchange_rows)
exchange_df["rate_id"] = range(1, len(exchange_df)+1)
path = f"{LANDING}/exchange_rates"; shutil.rmtree(path, ignore_errors=True)
write_to_landing(exchange_df, "exchange_rates", partition_cols=["year", "month", "day"])

# =============================================================================
# 2. CLICKSTREAM (200K vectorized)
# =============================================================================
print("\n2. Clickstream (200K)...")
N_CLICKS = 200_000
year_probs = np.array([YOY_MULTIPLIER.get(y, 1.0) for y in range(2020, 2027)])
year_probs /= year_probs.sum()
years = np.random.choice(range(2020, 2027), N_CLICKS, p=year_probs)
months = np.random.randint(1, 13, N_CLICKS)
months[years == 2026] = np.random.randint(1, 5, (years == 2026).sum())
days = np.random.randint(1, 29, N_CLICKS)

clickstream_df = pd.DataFrame({
    "event_id": [f"EVT_{i:08d}" for i in range(1, N_CLICKS+1)],
    "session_id": [f"SES_{np.random.randint(1000000,9999999)}" for _ in range(N_CLICKS)],
    "customer_id": np.where(np.random.random(N_CLICKS)<0.7, np.random.choice(ALL_CUSTOMER_IDS, N_CLICKS), None),
    "event_type": np.random.choice(["page_view","product_view","add_to_cart","checkout_start","purchase","search","filter_apply","wishlist_add"], N_CLICKS, p=[0.35,0.25,0.15,0.08,0.05,0.07,0.03,0.02]),
    "event_timestamp": [f"{y}-{m:02d}-{d:02d}T{np.random.randint(0,24):02d}:{np.random.randint(0,60):02d}:{np.random.randint(0,60):02d}" for y,m,d in zip(years,months,days)],
    "page_url": np.random.choice(["/home","/category","/product","/cart","/checkout","/search","/account"], N_CLICKS),
    "product_id": np.where(np.random.random(N_CLICKS)<0.5, np.random.choice(ALL_PRODUCT_IDS, N_CLICKS), None),
    "device_type": np.random.choice(["mobile","desktop","tablet"], N_CLICKS, p=[0.65,0.25,0.10]),
    "browser": np.random.choice(["chrome","safari","edge","firefox","samsung_browser"], N_CLICKS),
    "referrer": np.random.choice(["direct","google","social","email","affiliate"], N_CLICKS, p=[0.30,0.35,0.20,0.10,0.05]),
    "year": years, "month": months, "day": days,
})
clickstream_df["event_id"] = inject_nulls(clickstream_df["event_id"], 0.01)
clickstream_df.loc[np.random.random(N_CLICKS)<0.02, "event_type"] = "INVALID_EVENT"
path = f"{LANDING}/clickstream"; shutil.rmtree(path, ignore_errors=True)
write_to_landing(clickstream_df, "clickstream", partition_cols=["year", "month", "day"])

# =============================================================================
# 3. SOCIAL MEDIA (vectorized - daily × 5 platforms)
# =============================================================================
print("\n3. Social media...")
social_platforms = ["instagram", "twitter", "snapchat", "tiktok", "facebook"]
n_days = len(date_range_pd)
n_social = n_days * len(social_platforms)

social_year_arr = np.repeat([d.year for d in date_range_pd], len(social_platforms))
social_month_arr = np.repeat([d.month for d in date_range_pd], len(social_platforms))
social_day_arr = np.repeat([d.day for d in date_range_pd], len(social_platforms))
social_date_strs = np.repeat([d.strftime("%Y-%m-%d") for d in date_range_pd], len(social_platforms))
social_plats = np.tile(social_platforms, n_days)
social_ym = np.array([YOY_MULTIPLIER.get(y, 1.0) for y in social_year_arr])
posts = np.random.randint(0, 5, n_social)
followers = (np.random.randint(10000, 100000, n_social) * social_ym).astype(int)

social_df = pd.DataFrame({
    "metric_id": range(1, n_social+1), "platform": social_plats,
    "metric_date": social_date_strs, "followers": followers,
    "following": np.random.randint(100, 500, n_social), "posts_count": posts,
    "likes": (posts * np.random.randint(100, 1000, n_social) * social_ym).astype(int),
    "comments": (posts * np.random.randint(10, 100, n_social)).astype(int),
    "shares": (posts * np.random.randint(5, 50, n_social)).astype(int),
    "engagement_rate": np.round(np.random.uniform(0.01, 0.08, n_social), 4),
    "reach": (followers * np.random.uniform(0.1, 0.3, n_social)).astype(int),
    "impressions": (followers * np.random.uniform(0.3, 0.8, n_social)).astype(int),
    "year": social_year_arr, "month": social_month_arr, "day": social_day_arr,
})
path = f"{LANDING}/social_media"; shutil.rmtree(path, ignore_errors=True)
write_to_landing(social_df, "social_media", partition_cols=["year", "month", "day"])

# =============================================================================
# 4. PUSH NOTIFICATIONS (vectorized)
# =============================================================================
print("\n4. Push notifications...")
n_push = n_days * 2
push_idx = np.random.randint(0, n_days, n_push)
push_dates_pd = date_range_pd[push_idx]
push_year_arr = np.array([d.year for d in push_dates_pd])
push_ym = np.array([YOY_MULTIPLIER.get(y, 1.0) for y in push_year_arr])

sent = (np.random.randint(5000, 50000, n_push) * push_ym).astype(int)
delivered = (sent * np.random.uniform(0.85, 0.98, n_push)).astype(int)
opened = (delivered * np.random.uniform(0.10, 0.30, n_push)).astype(int)
clicked = (opened * np.random.uniform(0.20, 0.50, n_push)).astype(int)

push_df = pd.DataFrame({
    "notification_id": [f"PUSH_{i:07d}" for i in range(1, n_push+1)],
    "notification_type": np.random.choice(["promotional","transactional","reminder","personalized"], n_push, p=[0.40,0.25,0.20,0.15]),
    "campaign_name": [f"Campaign_{d.strftime('%Y%m%d')}_{i}" for i, d in enumerate(push_dates_pd, 1)],
    "sent_date": [d.strftime("%Y-%m-%d") for d in push_dates_pd],
    "sent_count": sent, "delivered_count": delivered, "opened_count": opened, "clicked_count": clicked,
    "delivery_rate": np.round(delivered/np.maximum(sent,1), 4),
    "open_rate": np.round(opened/np.maximum(delivered,1), 4),
    "click_rate": np.round(clicked/np.maximum(opened,1), 4),
    "target_segment": np.random.choice(["all","active","dormant","high_value","new"], n_push),
    "year": push_year_arr,
    "month": np.array([d.month for d in push_dates_pd]),
    "day": np.array([d.day for d in push_dates_pd]),
})
push_df["notification_id"] = inject_nulls(push_df["notification_id"], 0.02)
push_df.loc[np.random.random(n_push)<0.03, "notification_type"] = "INVALID"
path = f"{LANDING}/push_notifications"; shutil.rmtree(path, ignore_errors=True)
write_to_landing(push_df, "push_notifications", partition_cols=["year", "month", "day"])

print(f"\n\u2705 Supporting tables complete")
print(f"   Exchange rates: {len(exchange_df):,}")
print(f"   Clickstream: {len(clickstream_df):,}")
print(f"   Social media: {len(social_df):,}")
print(f"   Push notifications: {len(push_df):,}")

In [0]:
# =============================================================================
# SUMMARY & DATA QUALITY ESTIMATES
# =============================================================================

print("="*70)
print("SALLA E-COMMERCE DATA GENERATION COMPLETE")
print("="*70)

print("\n\U0001f4ca DATA VOLUMES GENERATED:")
print("-"*50)
print(f"  Customers:            {N_CUSTOMERS:>12,} rows")
print(f"  Products:             {N_PRODUCTS:>12,} rows")
print(f"  Stores:               {N_STORES:>12,} rows")
print(f"  Sales Orders:         {N_ORDERS:>12,} rows")
print(f"  Payment Transactions: {N_ORDERS:>12,} rows")
print(f"  Shipping Details:     {N_ORDERS:>12,} rows")
print(f"  Product Reviews:      {N_REVIEWS:>12,} rows")
print(f"  Customer Returns:     {N_RETURNS:>12,} rows")
print(f"  Ad Spend:             {len(ad_spend_df):>12,} rows")
print(f"  Competitor Pricing:   {len(competitor_df):>12,} rows")
print(f"  Exchange Rates:       {len(exchange_df):>12,} rows")
print(f"  Clickstream:          {len(clickstream_df):>12,} rows")
print(f"  Social Media:         {len(social_df):>12,} rows")
print(f"  Push Notifications:   {len(push_df):>12,} rows")
print("-"*50)
total_rows = (N_CUSTOMERS + N_PRODUCTS + N_STORES + N_ORDERS*3 + N_REVIEWS + N_RETURNS +
              len(ad_spend_df) + len(competitor_df) + len(exchange_df) +
              len(clickstream_df) + len(social_df) + len(push_df))
print(f"  TOTAL:                {total_rows:>12,} rows")

# Quick revenue check from Bronze
print("\n\n\U0001f4b0 REVENUE CHECK (from Bronze):")
print("-"*50)
rev_check = spark.sql("""
  SELECT COUNT(*) as orders,
    ROUND(SUM(CASE WHEN final_amount > 0 THEN final_amount ELSE 0 END)/1e9, 2) as revenue_B,
    ROUND(AVG(CASE WHEN final_amount > 0 THEN final_amount END), 2) as aov,
    ROUND(MAX(final_amount), 2) as max_order
  FROM salla_databricks.bronze.bronze_sales_orders
""").collect()[0]
print(f"  Total Orders:  {rev_check['orders']:,}")
print(f"  Revenue:       SAR {rev_check['revenue_B']}B")
print(f"  AOV:           SAR {rev_check['aov']}")
print(f"  Max Order:     SAR {rev_check['max_order']}")

print("\n  Orders per year:")
for year in range(2020, 2027):
    year_count = ORDERS_PER_YEAR.get(year, 0)
    print(f"    {year}: {year_count:>10,}")

print("\n\n\U0001f6e1\ufe0f EXPECTED DATA QUALITY (Silver Layer):")
print("-"*50)
print("  Table                     | Dropped  | Pass Rate")
print("  --------------------------|----------|----------")
print(f"  Customers (email issues)  | ~7,000   | ~93%")
print(f"  Payments (NULL pk)        | ~150,000 | ~97%")
print(f"  Shipping (NULL pk)        | ~150,000 | ~97%")
print(f"  Orders (soft warnings)    | ~0       | 100% (warn only)")
print(f"  Reviews (minor noise)     | ~10,000  | ~98%")
print(f"  Returns (minor noise)     | ~5,000   | ~98%")
print("-"*50)
print(f"  OVERALL EXPECTED PASS RATE: ~90-92%")

print("\n\n\U0001f680 NEXT STEPS:")
print("-"*50)
print("  1. Run Bronze pipeline (full refresh)")
print("  2. Run Silver pipeline (full refresh)")
print("  3. Run Gold pipeline (full refresh)")
print("  4. Refresh Power BI")
print("\n  Pipeline IDs:")
print("    Bronze: cd8c7c88-a926-47e5-9580-f1934c44a037")
print("    Silver: f5c1e5c9-8cc9-4c57-87fc-e0b40ba53803")
print("    Gold:   62279bb1-c0a1-4670-ad61-0e07b1184d71")
print("\n" + "="*70)
print("\u2705 DATA GENERATION COMPLETE")
print("="*70)